# alpha_lab quickstart

Walk-forward backtest of the example 12-1 cross-sectional momentum signal on the built-in
synthetic panel, entirely through the library API (the CLI equivalent is
`python scripts/run_backtest.py --config configs/base.yaml`).

**Read `CONVENTIONS.md` first** — it defines the timing law everything here obeys:
holdings `H_t = W_{t-lag}` with `lag >= 1`, truncation-invariant features, costs priced
on information available at execution time.

In [ ]:
import sys
from pathlib import Path

# project root on sys.path so the notebook works without `pip install -e .`:
# the directory that holds the alpha_lab PACKAGE, searched from the working
# directory upwards (notebooks/ or the project folder) and one level down
# (the folder that contains the project)
here = Path.cwd()
repo = next(
    (p for p in [here, *here.parents, here / "alpha_lab"] if (p / "alpha_lab" / "__init__.py").exists()),
    None,
)
if repo is None:
    raise RuntimeError(f"alpha_lab package not found from {here}; start in the project folder")
sys.path.insert(0, str(repo))

from alpha_lab.config import load_config
from alpha_lab.data import source_from_config, validate_market
from alpha_lab.features import FeatureStore
from alpha_lab.signals import build_signal
from alpha_lab.portfolio import from_config as portfolio_from_config
from alpha_lab.backtest import BacktestEngine, from_config as cost_from_config
from alpha_lab.risk import summary

# smaller panel than the demo config so the notebook is instant;
# dotted overrides are how sweeps work everywhere in alpha_lab
cfg = load_config(repo / "configs/base.yaml", overrides={"data.synthetic.n_days": 800})
cfg.signal

In [ ]:
data = source_from_config(cfg.data).load()
print(validate_market(data).summary())

signal = build_signal(cfg.signal)
features = FeatureStore().compute_all(list(signal.required_features), data)

engine = BacktestEngine(
    signal,
    portfolio_from_config(cfg.portfolio),
    cost_from_config(cfg.costs),
    cfg.backtest,
    portfolio_value=cfg.costs.portfolio_value,
)
result = engine.run(data, features)
result.windows

In [ ]:
metrics = summary(result)
{k: metrics[k] for k in (
    "sharpe_net", "sharpe_gross", "ann_return_net", "max_drawdown",
    "turnover_ann", "cost_drag_ann", "psr", "n_days", "n_windows",
)}

In [ ]:
import matplotlib.pyplot as plt

# walk-forward results are evaluated from the first test date on (CONVENTIONS.md,
# and what `summary` and the tearsheet do); the rows before it are the flat
# training prefix and would only draw a horizontal line
start = result.windows[0].test_start
equity = result.equity_curve().loc[start:]
turnover = result.turnover.loc[start:].rolling(21).mean()

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True, height_ratios=[3, 1])
axes[0].plot(equity.index, equity.values)
axes[0].set_title("net equity (walk-forward test period, after costs)")
axes[0].set_ylabel("NAV (start = 1)")
axes[1].plot(turnover.index, turnover.values)
axes[1].set_ylabel("21d turnover")
plt.tight_layout()
plt.show()

## Before trusting any new signal: run the leakage harness

`assert_signal_pit` recomputes the signal's features on truncated data at sampled
dates and raises `LookaheadError` if any score depends on the future. Make this a
reflex — a leaky backtest looks exactly like alpha until it meets live capital.

In [ ]:
from alpha_lab.testing import assert_signal_pit

assert_signal_pit(signal, data)  # raises LookaheadError on any peek
print("signal is point-in-time clean")

## Next steps

- Persist and index runs: `scripts/run_backtest.py` logs every run under `runs/`
  (config hash = experiment identity); inspect with
  `ExperimentTracker("../runs").list_runs()` and `.compare([...])`.
- New signal checklist lives at the bottom of `README.md` — features pass truncation
  invariance, `fit` sees train data only, PIT harness green, `execution_lag >= 1`,
  realistic costs, judged by deflated Sharpe given the trials you burned
  (`summary(result, n_trials=N)`, or `--n-trials N` on the command line).
- Real data: point `data.source: csv` at a directory with `close.csv` (+ optional
  `open/high/low/volume/unadjusted_close/universe.csv`) — `unadjusted_close` matters:
  all share math (commissions, ADV) uses it.